# Week 7 Extension Exercises: Integrated System Design

- These exercises build directly on the bookstore system from the starter notebook ([`order_service.py`](order_service.py) and [`inventory_service.py`](inventory_service.py)). 
- Work through them in order. Each one adds a real capability that a production system would need.

> Before starting, make sure ([`order_service.py`](order_service.py) and [`inventory_service.py`](inventory_service.py)) are in the same folder as this notebook.


In [1]:
import subprocess
import time
import requests

inventory_process = subprocess.Popen(["python3", "inventory_service.py"])
time.sleep(1.5)
order_process = subprocess.Popen(["python3", "order_service.py"])
time.sleep(1.5)

API_KEY = "week7-secret-key"
print(requests.get("http://localhost:5001/health").json())
print(requests.get("http://localhost:5000/health").json())

{'service': 'inventory_service', 'status': 'ok'}
{'service': 'order_service', 'status': 'ok'}


## Problem 1: Add a Notification Service

Right now the system has two services. Add a third: a **Notification Service** that logs a confirmation message whenever an order is successfully placed.

**Steps:**

1. Write a new file `notification_service.py` running on port `5002`, with one endpoint `POST /notify` that accepts a JSON body like `{"request_id": "...", "book_id": "...", "quantity": ...}` and logs a message such as `"order abc123 confirmed for book_001 x2"`.
2. Edit `order_service.py` so that after an order is confirmed (status 201), it also sends a request to the Notification Service.
3. If the Notification Service is unreachable, the order should still succeed. A failed notification should never block a confirmed order. Log a warning instead.

Write your `notification_service.py` in the cell below using `%%writefile`.


In [ ]:
%%writefile notification_service.py
# TODO: build the Notification Service here.
# It should:
#  - run on port 5002
#  - expose POST /notify
#  - log a confirmation message for each notification it receives
#  - use the same structured JSON log format as the other two services


Now edit `order_service.py` to call the Notification Service after a successful order. Think about where in `place_order()` that call belongs, and what should happen if the call fails.

Restart both services (and the new one) after you make changes, then test with the cell below.


In [ ]:
# TODO: restart order_service.py, inventory_service.py, and notification_service.py
# then place a test order here and confirm you see a notification log message.


## Problem 2: Request ID tracing across services

The Order Service already generates a short `request_id` for each order. Right now that ID only appears in the Order Service's own logs.

**Task:** pass the `request_id` through to the Inventory Service (and the Notification Service, if you completed Exercise 1) so that the *same* ID appears in the logs of all services involved in one order. This is called **distributed tracing**, and it is one of the main ways engineers debug a problem that spans several services.

**Steps:**

1. In `order_service.py`, include the `request_id` in the JSON body sent to `/check_stock`.
2. In `inventory_service.py`, read that `request_id` and include it in the log message.
3. Place an order and confirm the same ID shows up in both services' log output.


In [ ]:
# TODO: modify order_service.py and inventory_service.py, restart both,
# then place a test order and check that the same request_id appears in
# both services' log output.


## Exercise 3: Add a simple rate limiter

Right now a single client could send unlimited requests to the Order Service. Add a basic rate limiter: **no more than 5 requests per client within a 10-second window**.

**Hints:**

- You can identify a "client" here by the API key, since that is the only identity we have.
- Keep a dictionary that stores, per API key, a list of the timestamps of recent requests.
- On each request, drop timestamps older than 10 seconds, then check if there are already 5 or more remaining. If so, return a `429 Too Many Requests` response instead of processing the order.

Write a test below that sends 7 requests quickly and confirms that at least one of them comes back with status 429.


In [ ]:
# TODO: add rate limiting logic to order_service.py, restart it, then test here.

def send_test_order():
    return requests.post(
        "http://localhost:5000/order",
        json={"book_id": "book_001", "quantity": 1},
        headers={"X-API-Key": API_KEY},
    ).status_code

results = [send_test_order() for _ in range(7)]
print(results)
assert 429 in results, "Expected at least one request to be rate limited"


## Take-home Assignment: Decouple the services with a queue 

So far the Order Service calls the Inventory Service directly and waits for a reply. This means if the Inventory Service is slow, the Order Service is slow too. A common way to loosen this connection is to put a **queue** between them.

This exercise stays inside a single Python process, using the standard library `queue` module, so you can focus on the idea rather than on installing a message broker.

**Task:** inside this notebook (not as a separate service), build a small demo where:

1. A `queue.Queue` holds incoming orders.
2. A "producer" function adds orders to the queue, the same way the Order Service currently sends them.
3. A background thread (the "consumer") pulls orders off the queue one at a time and processes them (call `check_stock` in `inventory_service.py`, or simulate it).
4. The producer does not wait for the consumer. It can keep adding orders to the queue immediately.

> Once you have this working, write two or three sentences (in a markdown cell) comparing this queue-based approach to the direct HTTP call used in the rest of the notebook. When would you choose one over the other?


In [ ]:
import queue
import threading

order_queue = queue.Queue()

def consumer():
    # TODO: pull orders from order_queue in a loop and process each one
    pass

def producer(orders):
    # TODO: put each order onto order_queue without waiting for it to be processed
    pass

# TODO: start the consumer in a background thread, then call producer()
# with a few sample orders and observe that producer() returns immediately.


**Your comparison (edit this cell):**

_Write your answer here._


## Shutting down

Run this cell when you are finished.


In [ ]:
order_process.terminate()
inventory_process.terminate()
try:
    notification_process.terminate()
except NameError:
    pass
print("Services stopped.")